In [8]:
# not with target but serial
import serial, time
import chipwhisperer as cw, time

scope = cw.scope()
scope.default_setup()

# pins must stay in serial mode so the CDC port is routed to them
scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"

target = cw.target(scope, cw.targets.SimpleSerial)
target.ser.cwlite_usart.init(baud=115200, stopbits=1, parity="even")

ser = serial.Serial("/dev/ttyACM0", 115200, parity=serial.PARITY_EVEN,                    stopbits=1, bytesize=8, timeout=1)

# enter the bootloader: BOOT0 high, then reset
scope.io.pdic = "high"
scope.io.nrst = "low";  time.sleep(0.1)
scope.io.nrst = "high_z"; time.sleep(0.3)

ser.reset_input_buffer()          # drop any glitch bytes from the reset
ser.write(b"\x7f")
print("handshake:", ser.read(1).hex())          # expect 79

# Get ID: 0x02 + complement 0xFD
ser.write(bytes([0x02, 0xFD]))
print("get id:", ser.read(5).hex())             # expect 79 01 0435 79  # iF is NAK

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

In [4]:
import serial, time
import chipwhisperer as cw

scope = cw.scope()
scope.default_setup()
scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"

target = cw.target(scope, cw.targets.SimpleSerial)
target.ser.cwlite_usart.init(baud=115200, stopbits=1, parity="even")

ser = serial.Serial("/dev/ttyACM0", 115200, parity=serial.PARITY_EVEN, stopbits=1, bytesize=8, timeout=1)

def enter_bootloader():
    scope.io.pdic = "high"                 # BOOT0 high
    time.sleep(0.05)
    scope.io.nrst = "low";  time.sleep(0.2)
    scope.io.nrst = "high_z"; time.sleep(0.5)
    ser.reset_input_buffer()
    ser.write(b"\x7f")
    return ser.read(1)

def command(code):
    ser.reset_input_buffer()
    time.sleep(0.02)
    ser.write(bytes([code, code ^ 0xFF]))
    return ser.read(1)

for attempt in range(5):
    hs = enter_bootloader()
    print(attempt, "handshake:", hs.hex())
    if hs != b"\x79":
        continue
    time.sleep(0.05)
    ack = command(0x02)
    print("  get id ack:", ack.hex())
    if ack == b"\x79":
        n = ser.read(1)[0]
        print("  id:", ser.read(n + 1).hex(), ser.read(1).hex())
        break

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

In [12]:
import serial, time
import chipwhisperer as cw

scope = cw.scope()
scope.default_setup()
target = cw.target(scope, cw.targets.SimpleSerial)

scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"
target.ser.cwlite_usart.init(baud=115200, stopbits=1, parity="even")

ser = serial.Serial("/dev/ttyACM0", 115200, parity=serial.PARITY_EVEN, stopbits=1, bytesize=8, timeout=1)

scope.io.pdic = "high"; time.sleep(0.05)
scope.io.nrst = "low";  time.sleep(0.2)
scope.io.nrst = "high_z"; time.sleep(0.5)
ser.reset_input_buffer()
ser.write(b"\x7f");           print("hs :", ser.read(1).hex())   # hs : 79
ser.write(bytes([0x02, 0xFD])); print("id :", ser.read(5).hex()) # id : 7901046279

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

In [13]:
def cmd(code):
    ser.write(bytes([code, code ^ 0xFF]))
    return ser.read(1)

ser.reset_input_buffer()

# Get: bootloader version and supported commands
print(cmd(0x00).hex())          # 79
n = ser.read(1)[0]              # number of bytes that follow, minus 1
print(ser.read(n + 1).hex(), ser.read(1).hex())   # version, command list, ACK

# Read memory (0x11): address (4 bytes + XOR checksum), then length-1 + complement
def read_mem(addr, length):
    assert cmd(0x11) == b"\x79"
    a = addr.to_bytes(4, "big")
    ser.write(a + bytes([a[0] ^ a[1] ^ a[2] ^ a[3]]))
    if ser.read(1) != b"\x79":
        raise RuntimeError("address NACK (readout protection?)")
    ser.write(bytes([length - 1, (length - 1) ^ 0xFF]))
    if ser.read(1) != b"\x79":
        raise RuntimeError("length NACK")
    return ser.read(length)

print(read_mem(0x08000000, 16).hex())   # first 16 bytes of flash

79
310001021121314463738292 79
00000220b51500089d170008ab170008


In [15]:
# 1f is a real NACK, which fits the parity explanation.
# timeout means nothing came back. Check the wiring, or the reset may not have happened (see Step 3).
# Anything else is leftover data or noise.
    
enter_bootloader()
ser.write(b"\x00\xff")
r = ser.read(1)
print(r, r.hex() if r else "timeout")

b'' timeout


In [14]:
"""Dump the STM32L452 flash through the system bootloader (USART, 8E1)
using a ChipWhisperer Husky.

Wiring: Husky TIO1 (pin 10) / TIO2 (pin 12) to the Nucleo's PA9/PA10 so that
PA9 (Nucleo TX) lands on the pin set to serial_rx. PDIC (pin 13) -> BOOT0,
nRST (pin 5) -> NRST, common GND.

"""
import sys
import time

import serial
import chipwhisperer as cw

FLASH_BASE = 0x08000000
FLASH_SIZE = 512 * 1024      # STM32L452RE: 512 KB
CHUNK = 256                  # max bytes per Read Memory command
BAUD = 115200
RETRIES = 3

out_path = "flash_dump.bin"

scope = cw.scope()
scope.default_setup()
scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"

port = scope.get_serial_ports()[0]["port"]
ser = serial.Serial(port, BAUD, parity=serial.PARITY_EVEN,
                    stopbits=1, bytesize=8, timeout=1)


def enter_bootloader():
    """BOOT0 high, reset, then send 0x7F for auto-baud."""
    scope.io.pdic = "high"
    time.sleep(0.05)
    scope.io.nrst = "low"
    time.sleep(0.2)
    scope.io.nrst = "high_z"
    time.sleep(0.5)
    ser.reset_input_buffer()
    ser.write(b"\x7f")
    if ser.read(1) != b"\x79":
        raise RuntimeError("no ACK to 0x7F handshake")


def cmd(code):
    # No buffer flush here: it was NACKed in testing. Resync is done by
    # re-entering the bootloader instead (see read_chunk_with_retry).
    ser.write(bytes([code, code ^ 0xFF]))
    return ser.read(1)


def read_mem(addr, length):
    """Read Memory command (0x11), length 1..256."""
    if cmd(0x11) != b"\x79":
        raise RuntimeError("Read Memory NACK (readout protection enabled?)")
    a = addr.to_bytes(4, "big")
    ser.write(a + bytes([a[0] ^ a[1] ^ a[2] ^ a[3]]))
    if ser.read(1) != b"\x79":
        raise RuntimeError(f"address NACK at 0x{addr:08X}")
    ser.write(bytes([length - 1, (length - 1) ^ 0xFF]))
    if ser.read(1) != b"\x79":
        raise RuntimeError("length NACK")
    data = ser.read(length)
    if len(data) != length:
        raise RuntimeError(f"short read at 0x{addr:08X}: {len(data)}/{length}")
    return data


def read_chunk_with_retry(addr, length):
    for attempt in range(RETRIES):
        try:
            return read_mem(addr, length)
        except RuntimeError as e:
            print(f"\n  retry {attempt + 1}/{RETRIES}: {e}")
            enter_bootloader()        # clean state, then try again
    raise RuntimeError(f"giving up at 0x{addr:08X}")


enter_bootloader()

# Get: bootloader version and supported commands (same sequence that worked
# in manual testing)
if cmd(0x00) != b"\x79":
    raise RuntimeError("Get command NACK")
n = ser.read(1)[0]
info = ser.read(n + 1)
ser.read(1)  # final ACK
print("bootloader version:", hex(info[0]), "commands:", info[1:].hex())

t0 = time.time()
with open(out_path, "wb") as f:
    for off in range(0, FLASH_SIZE, CHUNK):
        data = read_chunk_with_retry(FLASH_BASE + off, CHUNK)
        f.write(data)
        if (off // CHUNK) % 32 == 0:
            pct = 100 * off / FLASH_SIZE
            print(f"\r{pct:5.1f}%  0x{FLASH_BASE + off:08X}", end="", flush=True)

print(f"\ndone: {FLASH_SIZE} bytes -> {out_path} in {time.time() - t0:.0f}s")

# Boot normally again next time
scope.io.pdic = "low"
ser.close()


scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

RuntimeError: Get command NACK

In [5]:
"""Dump the STM32L452 flash through the system bootloader (USART, 8E1)
using a ChipWhisperer Husky.

Wiring: Husky TIO1 (pin 10) / TIO2 (pin 12) to the Nucleo's PA9/PA10 so that
PA9 (Nucleo TX) lands on the pin set to serial_rx. PDIC (pin 13) -> BOOT0,
nRST (pin 5) -> NRST, common GND.

"""
import sys
import time

import serial
import chipwhisperer as cw

FLASH_BASE = 0x08000000
FLASH_SIZE = 512 * 1024      # STM32L452RE: 512 KB
CHUNK = 256                  # max bytes per Read Memory command
BAUD = 115200
RETRIES = 3

out_path = "flash_dump.bin"

scope = cw.scope()
scope.default_setup()
scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"

port = scope.get_serial_ports()[0]["port"]
ser = serial.Serial(port, BAUD, parity=serial.PARITY_EVEN,
                    stopbits=1, bytesize=8, timeout=1)


def enter_bootloader():
    """BOOT0 high, reset, then send 0x7F for auto-baud."""
    scope.io.pdic = "high"
    time.sleep(0.05)
    scope.io.nrst = "low"
    time.sleep(0.2)
    scope.io.nrst = "high_z"
    time.sleep(0.5)
    ser.reset_input_buffer()
    ser.write(b"\x7f")
    if ser.read(1) != b"\x79":
        raise RuntimeError("no ACK to 0x7F handshake")
    print("enter bootloader ok")


def cmd(code):
    ser.reset_input_buffer()          # resync: drop leftovers
    ser.write(bytes([code, code ^ 0xFF]))
    return ser.read(1)


def read_mem(addr, length):
    """Read Memory command (0x11), length 1..256."""
    if cmd(0x11) != b"\x79":
        raise RuntimeError("Read Memory NACK (readout protection enabled?)")
    a = addr.to_bytes(4, "big")
    ser.write(a + bytes([a[0] ^ a[1] ^ a[2] ^ a[3]]))
    if ser.read(1) != b"\x79":
        raise RuntimeError(f"address NACK at 0x{addr:08X}")
    ser.write(bytes([length - 1, (length - 1) ^ 0xFF]))
    if ser.read(1) != b"\x79":
        raise RuntimeError("length NACK")
    data = ser.read(length)
    if len(data) != length:
        raise RuntimeError(f"short read at 0x{addr:08X}: {len(data)}/{length}")
    return data


def read_chunk_with_retry(addr, length):
    for attempt in range(RETRIES):
        try:
            return read_mem(addr, length)
        except RuntimeError as e:
            print(f"\n  retry {attempt + 1}/{RETRIES}: {e}")
            enter_bootloader()        # clean state, then try again
    raise RuntimeError(f"giving up at 0x{addr:08X}")


enter_bootloader()

# Get ID as a sanity check (expect 79 01 04 62 79 for the L45x family)
if cmd(0x02) == b"\x79":
    n = ser.read(1)[0]
    pid = ser.read(n + 1)
    ser.read(1)
    print("product ID:", pid.hex())

t0 = time.time()
with open(out_path, "wb") as f:
    for off in range(0, FLASH_SIZE, CHUNK):
        data = read_chunk_with_retry(FLASH_BASE + off, CHUNK)
        f.write(data)
        if (off // CHUNK) % 32 == 0:
            pct = 100 * off / FLASH_SIZE
            print(f"\r{pct:5.1f}%  0x{FLASH_BASE + off:08X}", end="", flush=True)

print(f"\ndone: {FLASH_SIZE} bytes -> {out_path} in {time.time() - t0:.0f}s")

# Boot normally again next time
scope.io.pdic = "low"
ser.close()

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

RuntimeError: giving up at 0x08000000